In [1]:
import numpy as np
import pandas as pd

## Set global parameters

In [2]:
rng = np.random.default_rng(42)

N_STUDENTS = 8000
WEEKS = np.arange(1, 13)         # 1..12
POST_START = 7                   # weeks 7..12 are post
N_ASSIGNMENTS = 5                # per week

# SES distribution
ses_levels = ["low", "mid", "high"]
ses_probs = [0.30, 0.45, 0.25]

# Grades
grades = np.arange(6, 13)

## Create student table

In [3]:
students = pd.DataFrame({
    "student_id": np.arange(1, N_STUDENTS + 1),
    "grade_level": rng.choice(grades, size=N_STUDENTS, replace=True),
    "ses_band": rng.choice(ses_levels, size=N_STUDENTS, p=ses_probs, replace=True),
})

# Baseline score: Normal(70,10) clipped [40,95]
baseline_score = rng.normal(70, 10, size=N_STUDENTS)
students["baseline_score"] = np.clip(baseline_score, 40, 95)

# Baseline engagement: Normal(3.5,1.0) clipped [0,7]
baseline_eng = rng.normal(3.5, 1.0, size=N_STUDENTS)
students["baseline_engagement"] = np.clip(baseline_eng, 0, 7)

# Make baseline engagement correlate modestly with baseline score and SES
ses_shift = students["ses_band"].map({"low": -0.3, "mid": 0.0, "high": 0.3}).to_numpy()
students["baseline_engagement"] = np.clip(
    students["baseline_engagement"] + 0.02*(students["baseline_score"]-70) + ses_shift,
    0, 7
)

## Stratified randomization (grade_level × ses_band)

In [4]:
# Within each grade × SES group, randomly assign half the students to treatment
students["stratum"] = students["grade_level"].astype(str) + "_" + students["ses_band"].astype(str)
students["treatment"] = False

for stratum, idx in students.groupby("stratum").groups.items():
    idx = np.array(list(idx))
    rng.shuffle(idx)
    half = len(idx) // 2
    treat_idx = idx[:half]
    students.loc[treat_idx, "treatment"] = True

## Expand to student-week panel

In [5]:
# One row per student per week (8,000 students × 12 weeks)
panel = students.loc[:, ["student_id", "grade_level", "ses_band", "baseline_score", "baseline_engagement", "treatment"]].merge(
    pd.DataFrame({"week": WEEKS}),
    how="cross"
)

panel["post_period"] = panel["week"] >= POST_START

## Generate weekly engagement outcomes

In [6]:
# Engagement drops 0.05 active days per week after week 1
fatigue = -0.05 * (panel["week"] - 1)

# SES also shifts weekly engagement, on top of its effect on baseline
ses_eng = panel["ses_band"].map({"low": -0.2, "mid": 0.0, "high": 0.2}).to_numpy()

# Baseline-engagement as mean driver for active days
mu_active = panel["baseline_engagement"].to_numpy() + fatigue + ses_eng

# Treatment effect: only post; stronger for low baseline engagement, ceiling effects
low_base = (panel["baseline_engagement"] < 3.0).to_numpy()
high_base = (panel["baseline_engagement"] > 5.5).to_numpy()

treat = panel["treatment"].to_numpy()
post = panel["post_period"].to_numpy()

treat_effect = np.zeros(len(panel))
treat_effect += post * treat * 0.30
treat_effect += post * treat * low_base * 0.30   # extra for low baseline => total 0.60
treat_effect -= post * treat * high_base * 0.20  # reduce for high baseline => total 0.10

mu_active = mu_active + treat_effect

# Draw whole-number active days from a Poisson distribution, capped at 7
# (Poisson needs a positive rate, so the expected value is floored at 0.1)
lam = np.clip(mu_active, 0.1, 7.0)
active_days = rng.poisson(lam)
panel["weekly_active_days"] = np.clip(active_days, 0, 7)

# Assignments completed: Binomial(n=5, p depends on active days)
p_complete = np.clip(panel["weekly_active_days"] / 7 * 0.9 + 0.05, 0.02, 0.98)
panel["assignments_completed"] = rng.binomial(N_ASSIGNMENTS, p_complete)

# Engagement index: transparent composite
panel["engagement_index"] = 0.5 * panel["weekly_active_days"] + 0.5 * panel["assignments_completed"]

## Generate weekly assessment scores

In [7]:
# Pull scores toward the average: high-baseline students drift down, low-baseline drift up
rtm = -0.10 * (panel["baseline_score"] - 70)

# Week-to-week noise
score_noise = rng.normal(0, 4.0, size=len(panel))

# Steady learning growth for all students: +0.15 points per week
general_growth = panel["week"].to_numpy() * 0.15

# Treatment effect (post only): +1.2 base, +1.8 for low baseline engagement
score_treat = post * treat * (1.2 + 0.6 * low_base)  # avg ≈ 1.4

score = panel["baseline_score"].to_numpy() + rtm + general_growth + score_treat + score_noise
panel["assessment_score"] = np.clip(score, 0, 100)

## Simulate missing weeks

In [8]:
# Chance a student-week is missing depends on SES: 8% low, 5% mid, 3% high
miss_prob = panel["ses_band"].map({"low": 0.08, "mid": 0.05, "high": 0.03}).to_numpy()
missing = rng.random(len(panel)) < miss_prob
panel = panel.loc[~missing].reset_index(drop=True)

## Save dataset

In [9]:
# Save generated dataset
output_path = "data/synthetic_student_week_data.csv"
panel.to_csv(output_path, index=False)
print(f"Saved {len(panel):,} rows to {output_path}")

Saved 90,761 rows to data/synthetic_student_week_data.csv
